In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import rasterio

In [2]:
INVENTORY_PATH = Path("../Download/Quarter_mosaics/inventory.csv")
OUT_DIR = Path("../Download/Quarterly_NDVI")

In [3]:
inventory = pd.read_csv(INVENTORY_PATH, sep=';', header=0)
inventory = inventory.set_index(["Year", "Band"])

In [4]:
print(inventory.index)

MultiIndex([(2015,          'B04'),
            (2015,          'B08'),
            (2015, 'observations'),
            (2016,          'B04'),
            (2016,          'B08'),
            (2016, 'observations'),
            (2017,          'B04'),
            (2017,          'B08'),
            (2017, 'observations'),
            (2018,          'B04'),
            (2018,          'B08'),
            (2018, 'observations'),
            (2019,          'B04'),
            (2019,          'B08'),
            (2019, 'observations'),
            (2020,          'B04'),
            (2020,          'B08'),
            (2020, 'observations'),
            (2021,          'B04'),
            (2021,          'B08'),
            (2021, 'observations'),
            (2022,          'B04'),
            (2022,          'B08'),
            (2022, 'observations'),
            (2023,          'B04'),
            (2023,          'B08'),
            (2023, 'observations'),
            (2024,          

In [5]:
def Read_Band(year,band) -> np.ndarray:
    with rasterio.open(inventory.loc[year, band]["Path"], "r" ) as src:
        band = np.array(src.read())

        if len(band.shape) > 2 :
            if band.shape[0] == 1:
                band = band[0,:,:]
            else:
                raise BaseException(f"Cannot work with array of shape {band.shape}\n"
                                    f"   Incompatible shape encounterad at year {year} band {band} file {inventory.loc[year, band]["Path"]}")
    return band

def Calculate_ndvi(year):
    #year_bands = inventory[ (inventory["Year"] == year) ]

    red = Read_Band(year, "B04")
    nir = Read_Band(year, "B08")
    observations = Read_Band(year, "observations")

    red[red==-32768] = np.nan
    nir[nir==-32768] = np.nan

    red[red<0] = 0
    nir[nir<0] = 0 # because not harmonized data


    ndvi = (nir-red)/(nir+red)
    ndvi[red==-32768] = np.nan
    ndvi[nir==-32768] = np.nan
    ndvi[observations<1] = np.nan

    ndvi_clipped = np.clip(ndvi,-1,1) # just in case

    return ndvi_clipped, observations


In [9]:
out_inventory_path = OUT_DIR / "ndvi_inventory.csv"
out_inventory_path.parent.mkdir(parents=True, exist_ok=True)

# get coords, transform ...
with rasterio.open(inventory.iloc[0]["Path"], "r") as src:
    profile = src.profile
profile.update(count=2, dtype="float32", predctor=15)

with open(out_inventory_path, "w") as f:
  f.write("Year; Path;\n")

for year in inventory.index.unique(level="Year"):
    print(f"Calculating NDVI for {year}")
    ndvi, observations = Calculate_ndvi(year)

    print(f"   {np.nanmin(ndvi)} - {np.nanmax(ndvi)}   N/A-s {np.sum(np.isnan(ndvi))}" )

    raster = np.array([ndvi, observations])
    cur_tiff_path = OUT_DIR / f"{year}_NDVI.tif"

    with rasterio.open(cur_tiff_path, "w", **profile) as dst:
        dst.update_tags(DESCRIPTION=f"NDVI quarterly composite and number of available observation - year {year}")
        dst.write(raster)
    print(f"Saving geotiff {cur_tiff_path} \n")

    with open(out_inventory_path, "a") as f:
        f.write(f"{year};{cur_tiff_path.resolve()};\n")



Calculating NDVI for 2015
   -0.6000000238418579 - 1.0   N/A-s 11076
Saving geotiff ../Download/Quarter_NDVI/2015_NDVI.tif 

Calculating NDVI for 2016
   -0.3360995948314667 - 1.0   N/A-s 0
Saving geotiff ../Download/Quarter_NDVI/2016_NDVI.tif 

Calculating NDVI for 2017
   -0.7063491940498352 - 0.9693877696990967   N/A-s 10
Saving geotiff ../Download/Quarter_NDVI/2017_NDVI.tif 

Calculating NDVI for 2018
   -0.625668466091156 - 1.0   N/A-s 0
Saving geotiff ../Download/Quarter_NDVI/2018_NDVI.tif 

Calculating NDVI for 2019
   -0.4591549336910248 - 1.0   N/A-s 0
Saving geotiff ../Download/Quarter_NDVI/2019_NDVI.tif 

Calculating NDVI for 2020
   -0.8399999737739563 - 0.9798387289047241   N/A-s 0
Saving geotiff ../Download/Quarter_NDVI/2020_NDVI.tif 

Calculating NDVI for 2021
   -1.0 - 0.9606278538703918   N/A-s 0
Saving geotiff ../Download/Quarter_NDVI/2021_NDVI.tif 

Calculating NDVI for 2022
   -1.0 - 0.9780988097190857   N/A-s 126
Saving geotiff ../Download/Quarter_NDVI/2022_NDVI.ti